In [1]:
from mock_generator import MockGenerator
from numcosmo_py import Nc, Ncm, sky_match
import numpy as np

Ncm.cfg_init()
%load_ext autoreload
%autoreload 2

In [2]:
Omega_b = 0.0486
Omega_c = 0.2614
Omega_k = 0.0
H0 = 67.7

cosmo = Nc.HICosmoDEXcdm.new()
cosmo.omega_x2omega_k()
cosmo["Omegab"] = Omega_b
cosmo["Omegac"] = Omega_c
cosmo["Omegak"] = Omega_k
cosmo["H0"] = H0
cosmo["w"] = -1.0

prim = Nc.HIPrimPowerLaw.new()
prim.param_set_by_name("ln10e10ASA", 3.0)
prim.props.n_SA =  0.963 

reion = Nc.HIReionCamb.new()

cosmo.add_submodel(prim)
cosmo.add_submodel(reion)

tf = Nc.TransferFuncEH()

psml = Nc.PowspecMLTransfer.new(tf)
psml.require_kmin(1.0e-6)
psml.require_kmax(1.0e3)


psf = Ncm.PowspecFilter.new(psml, Ncm.PowspecFilterType.TOPHAT)
psf.set_best_lnr0()
psf.prepare(cosmo)

old_amplitude = np.exp(prim.props.ln10e10ASA)
ln10e10ASA = np.log((0.8/ cosmo.sigma8(psf)) ** 2 * old_amplitude)
prim.param_set_desc("ln10e10ASA", {"lower-bound": 2.0,"upper-bound": 5.0,"scale": 1.0,"abstol": 1.0e-50,"fit": True,"value": float(ln10e10ASA)})

dist = Nc.Distance.new(2.0)
dist.prepare(cosmo)

mulf = Nc.MultiplicityFuncDespali.new()
mulf.set_mdef(Nc.MultiplicityFuncMassDef.VIRIAL)
hmf = Nc.HaloMassFunction.new(dist, psf, mulf)

cluster_m = Nc.ClusterMassAscaso(lnRichness_min = np.log(1) ,lnRichness_max = np.log(1000))
cluster_m.param_set_by_name("cut", np.log(1))
cluster_m.param_set_desc("mup0",{"fit": True,"value": 3.022142935})
cluster_m.param_set_desc("mup1",{"fit": True,"value": 1.25})
cluster_m.param_set_desc("mup2",{"fit": True,"value": 0.0})
cluster_m.param_set_desc("sigmap0",{"fit": True,"value": 0.5})
cluster_m.param_set_desc("sigmap1",{"fit": True,"value": 0.0})
cluster_m.param_set_desc("sigmap2",{"fit": True,"value": 0.0})
#cluster_m = Nc.ClusterMassNodist(lnM_min=np.log(1e13), lnM_max=np.log(1e16))

In [3]:
class Completeness_model:
    
    """Class to generate completeness model for halo/cluster mocks."""
    def __init__(self, a):
        self.a = a

    
    def complete(self, mass, z):
        return np.ones_like(mass)

    def incomplete(self, mass, z):
        return  np.full_like(mass, self.a, dtype=float)
    
comp= Completeness_model(0.6)
mock_test = MockGenerator(cosmo=cosmo, hmf=hmf, cluster_m=cluster_m)
print(mock_test.sky_area())
halos = mock_test.generate_halos_from_hmf(comp.incomplete)
print(len(halos['z']))
print(len(halos[halos['is_detected'] == 1]['z']))
print(len(halos[halos['is_detected'] == 0]['z']))

401.93756131445895
8475
5091
3384


In [4]:
halos

halo_id,RA,DEC,z,Mass,Mass_obs,R200c,x1,x2,x3,halo_r,is_detected
int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64
200000,-2.50919762305275,7.8708132431071185,0.26749948175149685,30.75192134831244,0.002365777747078579,0.6302322176231189,1096.7163125797729,-48.06005971688721,151.75807921105869,1108.2088946694096,0
200001,9.014286128198322,-0.49353714484500477,0.43996890595334553,30.943914010908834,0.03682734751585276,0.6852737581781126,1719.0030128652838,272.7027157442407,-14.99276828717504,1740.563963925233,0
200002,4.639878836228101,8.892884299372394,0.2647226033605094,31.902432833015883,0.3673611664951392,0.9244460343573726,1080.7567600343168,87.71276137616908,169.66049005484027,1097.5032504524784,1
200003,1.973169683940732,-5.691240146960101,0.3373261133100588,30.86700860969307,0.097356955475433,0.6608499159745808,1364.3337322919147,47.00392850079541,-136.0486135786137,1371.9056555101583,0
200005,-6.880109593275947,7.813586477215975,0.22073675458176775,31.78527912007248,0.4558711110350014,0.883087894765688,910.4827311941293,-109.85982481934958,125.84680452240302,925.6810482669283,1
200006,-8.83832775663601,6.935297567748593,0.33838605847694475,30.643461340160954,0.19742990826662743,0.6134729462445168,1349.543256068105,-209.84483636584318,166.12895966244648,1375.8272735292162,1
200007,7.323522915498703,9.850308427742021,0.3098425977588632,32.13621253604595,1.181125531098665,1.0054832948641979,1240.4601179445433,159.42426439949517,217.15805088171703,1269.3758384942803,1
200008,2.0223002348641756,3.5719954309133564,0.3395267193950016,31.797429835543877,0.5114073516693844,0.9013797424474398,1376.505947179961,48.6050608848898,85.98052519731144,1380.0448272714298,0
200009,4.161451555920909,8.723145519853295,0.4725680271078641,30.240558063065478,0.2753273033879293,0.5435981571255487,1826.7619023913542,132.9134010502935,281.0300554048411,1853.0253404225436,1


In [7]:
mock_test.cluster_mass_interval = [1,1000]
halos["Mass_obs"] = halos["Mass_obs"]
class Purity_model:
    
    """Class to generate completeness model for halo/cluster mocks."""
    def __init__(self, b):
        self.b = b

    
    def pure(self, mass, z):
        return np.ones_like(mass)

    def impure(self, mass, z):
        return  np.full_like(mass, self.b, dtype=float)

pur= Purity_model(0.9)
clusters = mock_test.generate_clusters_from_halos(halos,2.0,pur.impure)
clusters

cluster_id,RA,DEC,z,Mass,R200c,x1,x2,x3,cluster_r,parent_id
int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64
100000,4.596554436817979,8.819765831332987,0.2648219544054689,0.3673611664951392,2.5160458969477725e-05,1081.424365913107,86.94388487726059,168.27660107445232,1097.886566443586,200002
100001,-6.8982001064029,7.866703652715395,0.22086550290488638,0.4558711110350014,2.5740481223734246e-05,910.8419952438768,-110.19496051450072,126.69695350703651,926.190146595505,200005
100002,-8.76670712031927,6.9411655383042055,0.3378708844981355,0.19742990826662743,2.400239242736222e-05,1347.890272450124,-207.8625213836558,166.26883371207776,1373.9215186558565,200006
100003,7.322935853118499,9.789120142377763,0.30936229231413714,1.181125531098665,3.320018358807515e-05,1238.8729567099806,159.20737865987184,215.82229818628005,1267.5695076332047,200007
100004,4.142361822775279,8.72874660238066,0.4730823365840787,0.2753273033879293,2.496808400723153e-05,1828.551328320285,132.4311531879704,281.2091054955541,1854.7820712014077,200009
100005,9.360342532496691,6.7943302814458555,0.2895463444041609,0.6484629219671506,2.7727068449300458e-05,1168.472073463692,192.60846525709582,141.0444587099702,1192.609972575285,200011
100006,-5.7844038300884995,-2.292674313037919,0.444574516661496,0.06430650367437656,2.3214732588738836e-05,1746.2449823465429,-176.8968700001823,-70.21426930170887,1756.5859172269122,200013
100007,-6.3515088914884466,8.708779196268733,0.45532238626991917,0.9261396509437918,3.096993900006929e-05,1762.202982103224,-196.152709171386,271.84345689167696,1793.8043651701785,200014
100008,-6.268341685661009,-1.395246625113185,0.25008673907091256,1.2738696362041078,3.396338876487618e-05,1034.2718107467329,-113.6062623423252,-25.301263277509403,1040.8000361607203,200015
